# Purpose

This notebook validates translated RAGTruth structural integrity, measures lightweight generation-corruption signals, and identifies translation pairs excluded by the deterministic pre-training filter. It does not establish perfect semantic equivalence.

## Configuration

Paths are repository-relative. The notebook and the filtering script read the same QA thresholds and deterministic implementation.

In [ ]:
import json
import os
import random
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import yaml
try:
    from IPython.display import display
except ModuleNotFoundError:
    def display(value):
        print(value)

def find_repo_root(start=None):
    start = (start or Path.cwd()).resolve()
    for candidate in (start, *start.parents):
        if (candidate / 'pyproject.toml').is_file() and (candidate / 'src').is_dir():
            return candidate
    raise FileNotFoundError('Repository root was not found.')

def repository_path(root, value, label):
    path = Path(str(value))
    if path.is_absolute() or '..' in path.parts:
        raise ValueError(f'{label} must be repository-relative.')
    return root / path

ROOT = find_repo_root()
os.chdir(ROOT)
sys.path.insert(0, str(ROOT / 'src'))
from ragtruth_transfer.io_utils import read_jsonl
from ragtruth_transfer.translation_qa import (
    classify_translation_pair,
    stable_pair_id,
    structural_mismatch_counts,
)

SPLITS = ('train', 'validation', 'test')
requested = os.environ.get('NOTEBOOK_CONFIG')
local_config = ROOT / 'notebooks' / 'configs' / 'ragtruth_translation_qa.local.yaml'
CONFIG_PATH = Path(requested) if requested else (local_config if local_config.is_file() else ROOT / 'notebooks' / 'configs' / 'ragtruth_translation_qa.example.yaml')
if not CONFIG_PATH.is_absolute():
    CONFIG_PATH = ROOT / CONFIG_PATH
config = yaml.safe_load(CONFIG_PATH.read_text(encoding='utf-8'))
data_config = config['data']
qa_config = config['qa']
SOURCE_DIR = repository_path(ROOT, data_config['source_dir'], 'data.source_dir')
TRANSLATED_DIR = repository_path(ROOT, data_config['translated_dir'], 'data.translated_dir')
FILTERED_DIR = repository_path(ROOT, data_config.get('filtered_dir', 'data/processed/ragtruth_textual_pt_nllb_filtered'), 'data.filtered_dir')
MIN_RATIO = float(qa_config.get('min_length_ratio', 0.40))
MAX_RATIO = float(qa_config.get('max_length_ratio', 2.50))
REPETITION_MIN_RUN = int(qa_config.get('repetition_min_run', 4))
display(pd.DataFrame({'Setting': ['Configuration', 'Source directory', 'Translated directory', 'Filtered output directory'], 'Value': [str(CONFIG_PATH.relative_to(ROOT)), str(SOURCE_DIR.relative_to(ROOT)), str(TRANSLATED_DIR.relative_to(ROOT)), str(FILTERED_DIR.relative_to(ROOT))]}))

## Load Data

In [ ]:
source_rows = {split: read_jsonl(SOURCE_DIR / f'{split}.jsonl') for split in SPLITS}
translated_rows = {split: read_jsonl(TRANSLATED_DIR / f'{split}.jsonl') for split in SPLITS}
translated_manifest = json.loads((TRANSLATED_DIR / 'manifest.json').read_text(encoding='utf-8')) if (TRANSLATED_DIR / 'manifest.json').is_file() else {}
example_count = sum(len(source_rows[split]) for split in SPLITS)
display(pd.DataFrame([{'split': split, 'EN examples': len(source_rows[split]), 'PT examples': len(translated_rows[split])} for split in SPLITS]))
print(f'Examples: {example_count:,}')
print(f'Translated manifest status: {translated_manifest.get("status")}')

## Structural Audit

Only claims and valid evidence slots may differ. Masked evidence and all structural/metadata fields must remain unchanged.

In [ ]:
structural_rows = []
for split in SPLITS:
    source_split, translated_split = source_rows[split], translated_rows[split]
    mismatch_totals = {name: 0 for name in structural_mismatch_counts({}, {})}
    for source, translated in zip(source_split, translated_split):
        for name, value in structural_mismatch_counts(source, translated).items():
            mismatch_totals[name] += value
    structural_rows.append({'split': split, 'EN rows': len(source_split), 'PT rows': len(translated_split), **mismatch_totals})
structural_audit = pd.DataFrame(structural_rows)
structural_columns = list(structural_mismatch_counts({}, {}))
STRUCTURAL_PASS = bool((structural_audit[structural_columns].to_numpy() == 0).all()) and bool((structural_audit['EN rows'] == structural_audit['PT rows']).all())
display(structural_audit)
print(f'Structural audit: {"PASS" if STRUCTURAL_PASS else "FAIL"}')

## Translation QA

The reusable `translation_qa` module is the single source of truth for length, repetition, normalization, and exclusion classification.

In [ ]:
pair_rows = []
example_records = []
for split in SPLITS:
    for source, translated in zip(source_rows[split], translated_rows[split]):
        example_records.append((split, source, translated))
        pair_rows.append({'split': split, 'example_id': source['example_id'], 'source_id': source['source_id'], 'label': source['label'], 'field_type': 'claim', 'evidence_index': None, 'source_text': str(source.get('claim') or ''), 'translated_text': str(translated.get('claim') or '')})
        for index, is_valid in enumerate(source['evidence_mask']):
            if is_valid is True:
                pair_rows.append({'split': split, 'example_id': source['example_id'], 'source_id': source['source_id'], 'label': source['label'], 'field_type': 'evidence', 'evidence_index': index, 'source_text': str(source['evidence'][index] or ''), 'translated_text': str(translated['evidence'][index] or '')})
pairs = pd.DataFrame(pair_rows)
metrics_cache = {}
for source_text, translated_text in zip(pairs['source_text'], pairs['translated_text']):
    key = (source_text, translated_text)
    if key not in metrics_cache:
        metrics_cache[key] = classify_translation_pair(source_text, translated_text, min_length_ratio=MIN_RATIO, max_length_ratio=MAX_RATIO, repetition_min_run=REPETITION_MIN_RUN)
qa = pd.concat([pairs.reset_index(drop=True), pd.DataFrame([metrics_cache[(source, translated)] for source, translated in zip(pairs['source_text'], pairs['translated_text'])])], axis=1)

def unique_pair_count(frame, flag):
    return int(frame.loc[frame[flag], ['source_normalized', 'translated_normalized']].drop_duplicates().shape[0])

raw_unique_source_texts = pairs['source_text'].nunique()
raw_unique_translation_pairs = pairs[['source_text', 'translated_text']].drop_duplicates().shape[0]
normalized_unique_source_texts = qa['source_normalized'].nunique()
normalized_unique_translation_pairs = qa[['source_normalized', 'translated_normalized']].drop_duplicates().shape[0]
qa_summary = pd.DataFrame([
    {'measure': 'Total translation occurrences', 'value': len(qa)},
    {'measure': 'Raw unique source texts', 'value': raw_unique_source_texts},
    {'measure': 'Raw unique translation pairs', 'value': raw_unique_translation_pairs},
    {'measure': 'Normalized unique source texts', 'value': normalized_unique_source_texts},
    {'measure': 'Normalized unique translation pairs', 'value': normalized_unique_translation_pairs},
    {'measure': 'Empty translations', 'value': int(qa['empty_translation'].sum())},
    {'measure': 'Control-character issues', 'value': int(qa['control_character_issue'].sum())},
    {'measure': 'Low-ratio unique', 'value': unique_pair_count(qa, 'low_ratio')},
    {'measure': 'High-ratio unique', 'value': unique_pair_count(qa, 'high_ratio')},
    {'measure': 'High-confidence repetition unique', 'value': unique_pair_count(qa, 'high_confidence_repetition')},
    {'measure': 'Exclusion candidates unique', 'value': unique_pair_count(qa, 'exclusion_candidate')},
])
display(qa_summary)

## Length Overview

In [ ]:
ratios = qa['length_ratio'].dropna()
display(pd.DataFrame({'statistic': ['min', 'median', 'mean', 'p99', 'max'], 'length_ratio': [ratios.min(), ratios.median(), ratios.mean(), ratios.quantile(0.99), ratios.max()]}).style.format({'length_ratio': '{:.4f}'}))
figure, axis = plt.subplots(figsize=(7, 4))
axis.hist(ratios, bins=60, color='#2a9d8f', edgecolor='white')
axis.axvline(MIN_RATIO, color='#e76f51', linestyle='--', label='low-ratio threshold')
axis.axvline(MAX_RATIO, color='#e76f51', linestyle='--', label='high-ratio threshold')
axis.set_xlabel('Translated characters / source characters')
axis.set_ylabel('Translation occurrences')
axis.legend()
figure.tight_layout()
plt.show()

## Exclusion Candidates

The deterministic filter excludes an entire example when its claim or any valid evidence has `low_ratio OR high_ratio OR high_confidence_repetition`. Masked evidence does not participate.

In [ ]:
candidate_occurrences = qa.loc[qa['exclusion_candidate']].copy()
identity_columns = ['source_normalized', 'translated_normalized']
exclusion_candidates = (candidate_occurrences.groupby(identity_columns, as_index=False, sort=False)
    .agg(source_text=('source_text', 'first'), translated_text=('translated_text', 'first'), length_ratio=('length_ratio', 'first'), is_low_ratio=('low_ratio', 'any'), is_high_ratio=('high_ratio', 'any'), is_high_confidence_repetition=('high_confidence_repetition', 'any'), source_has_repetition=('source_has_repetition', 'first'), translation_has_repetition=('translation_has_repetition', 'first'), source_repetition_metric=('source_repetition_metric', 'first'), translation_repetition_metric=('translation_repetition_metric', 'first'), repetition_excess=('repetition_excess', 'first'), repetition_severity=('repetition_severity', 'first'), occurrence_count=('source_text', 'size'), split=('split', 'first'), example_id=('example_id', 'first'), source_id=('source_id', 'first'), label=('label', 'first'), field_type=('field_type', 'first'), evidence_index=('evidence_index', 'first')))
reason_order = [('is_high_confidence_repetition', 'high_confidence_repetition'), ('is_high_ratio', 'high_ratio'), ('is_low_ratio', 'low_ratio')]
exclusion_candidates['candidate_id'] = [stable_pair_id(source, translated) for source, translated in zip(exclusion_candidates['source_normalized'], exclusion_candidates['translated_normalized'])]
exclusion_candidates['candidate_reasons'] = exclusion_candidates.apply(lambda row: '|'.join(label for column, label in reason_order if bool(row[column])), axis=1)
exclusion_candidates['_priority'] = np.select([exclusion_candidates['is_high_confidence_repetition'], exclusion_candidates['is_high_ratio'], exclusion_candidates['is_low_ratio']], [0, 1, 2], default=9)
exclusion_candidates['_severity_sort'] = np.where(exclusion_candidates['is_high_confidence_repetition'], -exclusion_candidates['repetition_excess'], 0)
exclusion_candidates['_ratio_sort'] = np.select([exclusion_candidates['is_high_ratio'], exclusion_candidates['is_low_ratio']], [-exclusion_candidates['length_ratio'], exclusion_candidates['length_ratio']], default=0.0)
exclusion_candidates = exclusion_candidates.sort_values(['_priority', '_severity_sort', '_ratio_sort', 'candidate_id'], kind='mergesort').drop(columns=['_priority', '_severity_sort', '_ratio_sort']).reset_index(drop=True)
category_counts = {flag: unique_pair_count(qa, flag) for flag in ['low_ratio', 'high_ratio', 'high_confidence_repetition']}
overlap_count = sum(category_counts.values()) - len(exclusion_candidates)
assert len(exclusion_candidates) == int(qa_config.get('expected_candidate_translation_pairs', 216))
assert exclusion_candidates['candidate_id'].is_unique
print(f'Low-ratio: {category_counts["low_ratio"]}; high-ratio: {category_counts["high_ratio"]}; high-confidence repetition: {category_counts["high_confidence_repetition"]}')
print(f'Overlaps: {overlap_count}; exclusion candidates: {len(exclusion_candidates)}')
display(exclusion_candidates.head(10))

## Sanity Sample

The random sample shows complete examples and valid evidence slots. Automatic QA does not guarantee semantic equivalence. Literal, stylistically awkward, or mildly semantically imperfect translations are accepted as translation noise in the initial experiment.

In [ ]:
random_generator = random.Random(int(qa_config.get('random_seed', 42)))
sample_size = min(int(qa_config.get('random_examples', 30)), len(example_records))
sample_rows = []
for split, source, translated in random_generator.sample(example_records, sample_size):
    valid_indices = [index for index, valid in enumerate(source['evidence_mask']) if valid is True]
    sample_rows.append({'split': split, 'example_id': source['example_id'], 'label': source['label'], 'claim EN': source['claim'], 'claim PT': translated['claim'], 'valid evidence EN': '\n'.join(f'Evidence {index + 1}: {source["evidence"][index]}' for index in valid_indices), 'valid evidence PT': '\n'.join(f'Evidence {index + 1}: {translated["evidence"][index]}' for index in valid_indices)})
display(pd.DataFrame(sample_rows).style.set_properties(**{'white-space': 'pre-wrap', 'vertical-align': 'top', 'max-width': '520px'}))

## Final Summary

In [ ]:
qa_ready = STRUCTURAL_PASS and int(qa['empty_translation'].sum()) == 0 and int(qa['control_character_issue'].sum()) == 0 and len(exclusion_candidates) == int(qa_config.get('expected_candidate_translation_pairs', 216))
print(f'Structural audit: {"PASS" if STRUCTURAL_PASS else "FAIL"}')
print(f'Empty translations: {int(qa["empty_translation"].sum())}')
print(f'Control-character issues: {int(qa["control_character_issue"].sum())}')
print(f'Exclusion candidates: {len(exclusion_candidates)} unique translation pairs')
print(f'QA status: {"READY FOR DETERMINISTIC FILTERING" if qa_ready else "FAIL"}')
print('READY FOR DETERMINISTIC FILTERING does not imply semantic equivalence validation.')